<a href="https://colab.research.google.com/github/yg36/Get-Insights-from-any-dataset/blob/main/Approach_2_AUto_EDA_%2B_rule_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ucimlrepo
!pip install pandas
!pip install numpy
!pip install scipy
!pip install itertools
!pip install ydata-profiling
!pip install openai


ERROR: Could not find a version that satisfies the requirement itertools (from versions: none)
ERROR: No matching distribution found for itertools
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.8/400.8 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.7/679.7 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 6.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import skew, pearsonr
from itertools import combinations
from ydata_profiling import ProfileReport

/tmp/ipykernel_5276/4034909543.py:5: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


# **Dataset injection**

In [ ]:
def load_dataset(data):
    """
    Accepts:
    - filepath (csv)
    - pandas DataFrame
    """
    if isinstance(data, str):
        df = pd.read_csv(data)
    elif isinstance(data, pd.DataFrame):
        df = data.copy()
    else:
        raise ValueError("Unsupported data format")

    return df

# **AUTO EDA**

In [ ]:
def profile_dataset(df):
    profile = {}

    report = ProfileReport(X, title="Online Retail Profiling Report")
    report.to_notebook_iframe()

    profile['shape'] = df.shape
    profile['missing'] = df.isnull().sum().to_dict()
    profile['dtypes'] = df.dtypes.astype(str).to_dict()

    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

    profile['numeric_cols'] = numeric_cols
    profile['categorical_cols'] = categorical_cols

    profile['summary'] = df.describe(include='all').to_dict()

    return profile

# **STATISTICAL ANALYSIS**

In [ ]:

def detect_signals(df, profile):
    signals = []

    numeric_cols = profile['numeric_cols']
    categorical_cols = profile['categorical_cols']

    # --- Skewness (distribution signal)
    for col in numeric_cols:
        val = skew(df[col].dropna())
        if abs(val) > 1:
            signals.append({
                "type": "skewness",
                "column": col,
                "value": val
            })

    # --- Correlation signals
    for col1, col2 in combinations(numeric_cols, 2):
        try:
            corr, _ = pearsonr(df[col1].dropna(), df[col2].dropna())
            if abs(corr) > 0.7:
                signals.append({
                    "type": "correlation",
                    "columns": (col1, col2),
                    "value": corr
                })
        except:
            continue

    # --- Categorical impact (group differences)
    for cat in categorical_cols:
        for num in numeric_cols:
            grouped = df.groupby(cat)[num].mean()

            if len(grouped) > 1:
                variance = grouped.var()
                if variance > df[num].var() * 0.1:
                    signals.append({
                        "type": "category_impact",
                        "category": cat,
                        "numeric": num,
                        "variance": variance
                    })

    return signals

# **RULE ENGINE**

In [ ]:
def apply_rules(signals):
    rules_output = []

    for s in signals:

        if s['type'] == "skewness":
            if s['value'] > 1:
                rules_output.append(f"{s['column']} is heavily right-skewed")
            elif s['value'] < -1:
                rules_output.append(f"{s['column']} is heavily left-skewed")

        elif s['type'] == "correlation":
            rules_output.append(
                f"Strong correlation between {s['columns'][0]} and {s['columns'][1]} ({round(s['value'],2)})"
            )

        elif s['type'] == "category_impact":
            rules_output.append(
                f"{s['category']} significantly affects {s['numeric']}"
            )

    return rules_output

# **LLM BASED INSIGHT GENERATOR**

In [ ]:
def generate_insight_prompt(profile, rules_output):
    prompt = f"""
You are a data analyst.

Dataset Overview:
- Rows: {profile['shape'][0]}
- Columns: {profile['shape'][1]}

Key Observations:
{chr(10).join(rules_output)}

Convert this into business-friendly insights.
Explain patterns, risks, and opportunities.
Avoid technical jargon.
"""
    return prompt

In [ ]:
from google.colab import userdata
api_key = userdata.get("OPENAI_API_KEY")
gemini_api_key = userdata.get("GEMINIA_API_KEY")

In [ ]:
!pip install google

In [ ]:
from google import genai

In [ ]:
client = genai.Client(api_key=gemini_api_key)

def generate_insights_from_llm(prompt):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text

# **PIPELINE EXECUTION**

In [ ]:
def run_pipeline(data):

    df = load_dataset(data)

    profile = profile_dataset(df)

    signals = detect_signals(df, profile)

    rules_output = apply_rules(signals)

    insight_prompt = generate_insight_prompt(profile, rules_output)

    insights = generate_insights_from_llm(insight_prompt)

    return {
        "profile": profile,
        "signals": signals,
        "rules": rules_output,
        "llm_prompt": insight_prompt,
        "insights": insights
    }

# **USAGE**

In [ ]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
online_retail = fetch_ucirepo(id=352)

# data (as pandas dataframes)
X = online_retail.data.features
y = online_retail.data.targets

# metadata
print(online_retail.metadata)

# variable information
print(online_retail.variables)

{'uci_id': 352, 'name': 'Online Retail', 'repository_url': 'https://archive.ics.uci.edu/dataset/352/online+retail', 'data_url': 'https://archive.ics.uci.edu/static/public/352/data.csv', 'abstract': 'This is a transactional data set which contains all the transactions occurring between 01/12/2010 and 09/12/2011 for a UK-based and registered non-store online retail.', 'area': 'Business', 'tasks': ['Classification', 'Clustering'], 'characteristics': ['Multivariate', 'Sequential', 'Time-Series'], 'num_instances': 541909, 'num_features': 6, 'feature_types': ['Integer', 'Real'], 'demographics': [], 'target_col': None, 'index_col': ['InvoiceNo', 'StockCode'], 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2015, 'last_updated': 'Mon Oct 21 2024', 'dataset_doi': '10.24432/C5BW33', 'creators': ['Daqing Chen'], 'intro_paper': {'ID': 361, 'type': 'NATIVE', 'title': 'Data mining for the online retail industry: A case study of RFM model-based customer segmenta

In [ ]:
result = run_pipeline(X)

print("Rules are:" , result['rules'])
print("\nFinal insights:\n", result['insights'])

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 6/6 [00:11<00:00,  1.87s/it]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Rules are: ['UnitPrice is heavily right-skewed', 'Description significantly affects Quantity', 'Description significantly affects UnitPrice', 'Description significantly affects CustomerID', 'InvoiceDate significantly affects Quantity', 'InvoiceDate significantly affects UnitPrice', 'InvoiceDate significantly affects CustomerID', 'Country significantly affects CustomerID']

Final insights:
 Here's a business-friendly breakdown of your dataset's key observations, highlighting patterns, risks, and opportunities:

---

### Data Insights: Unlocking Our Business Potential

We've analyzed over half a million customer transactions and uncovered some compelling insights that can help us optimize our sales, marketing, and operations.

#### Key Patterns We've Discovered:

1.  **Our Product Catalog is Diverse in Price:** While the majority of our sales transactions are for more affordably priced items, we also successfully sell premium, higher-value products. This indicates a broad market appeal a